In [0]:
# Test — Day 6 Gold Layer
# Runs after gold_dlt_pipeline completes.

import unittest

dbutils.widgets.text("catalog_name", "vstone_catalog", "1. Catalog Name")
dbutils.widgets.text("gold_schema", "gold", "2. Gold Schema")
CATALOG = dbutils.widgets.get("catalog_name")
GOLD = dbutils.widgets.get("gold_schema")


class GoldDay6Tests(unittest.TestCase):

    def test_dim_date_covers_the_real_data_window(self):
        df = spark.table(f"{CATALOG}.{GOLD}.dim_date")
        in_range = df.filter("date_key BETWEEN '2023-06-02' AND '2024-03-11'").count()
        self.assertEqual(in_range, 284, f"Expected 284 days in the real data window, got {in_range}.")

    def test_dim_street_has_exactly_one_active_row_per_street(self):
        """SCD2 correctness: __END_AT IS NULL should give exactly 36 rows, no duplicates."""
        df = spark.table(f"{CATALOG}.{GOLD}.dim_street").filter("__END_AT IS NULL")
        total = df.count()
        distinct = df.select("street_id").distinct().count()
        self.assertEqual(total, 36, f"Expected 36 active streets, got {total}.")
        self.assertEqual(total, distinct, "Duplicate active street_id rows — SCD2 apply_changes is broken.")

    def test_dim_node_location_flags_location_7_instead_of_excluding_it(self):
        """
        FIXED premise: location=7 has bad (0,0) source coordinates, but it's
        a real sensor with real fact_traffic_counts data — excluding it from
        the dimension orphaned 1,626,982 fact rows. It's now present with
        NULL coordinates and has_valid_coordinates=false instead.
        """
        df = spark.table(f"{CATALOG}.{GOLD}.dim_node_location").filter("__END_AT IS NULL")
        locations = {r["location"] for r in df.select("location").collect()}
        self.assertEqual(len(locations), 14, f"Expected all 14 locations present, got {len(locations)}: {locations}")

        loc7 = df.filter("location = 7").collect()
        self.assertEqual(len(loc7), 1, "Expected exactly one active row for location=7.")
        self.assertFalse(loc7[0]["has_valid_coordinates"], "location=7 should be flagged has_valid_coordinates=false.")
        self.assertIsNone(loc7[0]["latitude"], "location=7's latitude should be NULL, not the misleading raw 0.0.")
        self.assertIsNone(loc7[0]["longitude"], "location=7's longitude should be NULL, not the misleading raw 0.0.")

    def test_fact_street_readings_fk_integrity(self):
        """Every street_id in the fact must exist in the active dim — no orphan FKs."""
        fact = spark.table(f"{CATALOG}.{GOLD}.fact_street_readings")
        dim = spark.table(f"{CATALOG}.{GOLD}.dim_street").filter("__END_AT IS NULL").select("street_id")
        orphans = fact.join(dim, on="street_id", how="left_anti").count()
        self.assertEqual(orphans, 0, f"{orphans} fact_street_readings rows have a street_id not in dim_street.")

    def test_fact_traffic_counts_grain_is_clean(self):
        """(location, reading_ts, reading_id) — corrected grain, see requirements_and_assumptions.md."""
        df = spark.table(f"{CATALOG}.{GOLD}.fact_traffic_counts")
        total = df.count()
        distinct = df.select("location", "reading_ts", "reading_id").distinct().count()
        self.assertEqual(total, distinct,
                          f"fact_traffic_counts grain not clean: {total:,} vs {distinct:,} distinct triples.")

    def test_fact_traffic_counts_fk_integrity(self):
        fact = spark.table(f"{CATALOG}.{GOLD}.fact_traffic_counts")
        dim = spark.table(f"{CATALOG}.{GOLD}.dim_node_location").filter("__END_AT IS NULL").select("location")
        orphans = fact.join(dim, on="location", how="left_anti").count()
        self.assertEqual(orphans, 0, f"{orphans} fact_traffic_counts rows have a location not in dim_node_location.")

    def test_fact_street_readings_raining_still_clipped(self):
        """Confirms Day 5's business rule survived into Gold."""
        df = spark.table(f"{CATALOG}.{GOLD}.fact_street_readings")
        out_of_range = df.filter("raining_clipped < 0 OR raining_clipped > 100").count()
        self.assertEqual(out_of_range, 0, f"{out_of_range} rows have raining_clipped outside [0,100] in Gold.")

    def test_top10_streets_has_exactly_10_rows(self):
        df = spark.table(f"{CATALOG}.{GOLD}.agg_top10_streets_by_pollution")
        self.assertEqual(df.count(), 10, f"Expected exactly 10 rows, got {df.count()}.")

    def test_top10_streets_is_actually_sorted_descending(self):
        rows = spark.table(f"{CATALOG}.{GOLD}.agg_top10_streets_by_pollution") \
            .select("avg_pollution").orderBy("avg_pollution", ascending=False).collect()
        values = [r["avg_pollution"] for r in rows]
        self.assertEqual(values, sorted(values, reverse=True), "agg_top10_streets_by_pollution is not sorted descending.")

    def test_all_gold_tables_have_gold_load_dt(self):
        tables = ["dim_date", "dim_street", "dim_node_location", "fact_street_readings",
                  "fact_traffic_counts", "fact_citizen_reports", "agg_monthly_street_trend",
                  "agg_top10_streets_by_pollution", "agg_traffic_by_location_daily",
                  "agg_citizen_reports_by_street"]
        for t in tables:
            with self.subTest(table=t):
                missing = spark.table(f"{CATALOG}.{GOLD}.{t}").filter("gold_load_dt IS NULL").count()
                self.assertEqual(missing, 0, f"{t} has {missing} rows missing gold_load_dt.")


if __name__ == "__main__":
    suite = unittest.TestLoader().loadTestsFromTestCase(GoldDay6Tests)
    result = unittest.TextTestRunner(verbosity=2).run(suite)
    if not result.wasSuccessful():
        raise Exception("Day 6 Gold tests FAILED — see output above.")
